## 依赖导入

本 notebook 汇总线性代数与线性规划建模的常用工具：`cvxpy` 建立并求解线性（整数）规划模型；`numpy` 与 `numpy.linalg`（含 `svd`）提供矩阵运算与奇异值分解，`scipy.sparse.linalg.eigs` 求解稀疏主特征值；`pandas` 读写数据；`PIL.Image` 读取图像；`matplotlib.pyplot` 负责绘图。

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from numpy import linalg as LA
from numpy.linalg import svd
from scipy.sparse.linalg import eigs

## 全局绘图配置

设置全局 `plt.rc`：字号 13、中文字体 SimHei，保证后续 PageRank 柱状图、SVD 图像压缩图等的中文标题与标签正常显示。

In [ ]:
plt.rc("font", size=13)
plt.rc("font", family="SimHei")

## 1.1

## 1.2 整数线性规划：两种产品的最大利润

用 `cvxpy` 建模两变量整数线性规划：产量 `x11=(x_1,x_2)` 为非负整数，单件利润 4000、3000，目标最大化 $4000x_1+3000x_2$；约束 $2x_1+x_2\le 10$、$x_1+x_2\le 8$、$x_2\le 7$。调用 `problem11.solve()` 求解并输出最优值与最优解。

In [ ]:
x11 = cp.Variable(2, nonneg=True, integer=True)

objective11 = cp.Maximize(x11[0] * 4000 + x11[1] * 3000)
constraints11 = [x11[0] * 2 + x11[1] <= 10, x11[0] + x11[1] <= 8, x11[1] <= 7]
problem11 = cp.Problem(objective11, constraints11)
problem11.solve()

print(f"Optimal value: {problem11.value}")
print(f"x = {x11.value}")

## 1.3 三变量线性规划

三个非负连续变量 $x_1,x_2,x_3$，目标最大化 $2x_1+3x_2-5x_3$；约束 $x_1+x_2+x_3=7$、$2x_1-5x_2+x_3\ge 10$、$x_1+3x_2+x_3\le 12$。用 `cp.Variable(nonneg=True)`、`cp.Maximize`、`cp.Problem.solve` 建模求解，输出保留 3 位小数的最优值与各变量取值。

In [ ]:
x1, x2, x3 = (
    cp.Variable(nonneg=True),
    cp.Variable(nonneg=True),
    cp.Variable(nonneg=True),
)
objective13 = cp.Maximize(x1 * 2 + x2 * 3 - x3 * 5)
constraints13 = [x1 + x2 + x3 == 7, x1 * 2 - x2 * 5 + x3 >= 10, x1 + x2 * 3 + x3 <= 12]
problem13 = cp.Problem(objective13, constraints13)
problem13.solve()
print(f"Optimal value: {problem13.value:.3f}")
print(f"x1 = {x1.value:.3f}\nx2 = {x2.value:.3f}\nx3 = {x3.value:.3f}")

## 1.4 整数规划：排班/采购安排

以 $4\times 4$ 非负整数矩阵 `x14` 表示各需求在各时段的安排量，目标最小化 $2800\sum x_{:,0}+4500\sum x_{:,1}+6000\sum x_{:,2}+7300\,x_{0,3}$；各时段累计需求给出若干 `>=` 覆盖约束，另有若干 `==0` 禁止项。用 `cvxpy` 求解，输出取整的最优值与解矩阵（`np.abs` 抹平求解器的极小负值噪声）。

In [ ]:
x14 = cp.Variable((4, 4), nonneg=True, integer=True)
objective14 = cp.Minimize(
    2800 * sum(x14[0:4, 0])
    + 4500 * sum(x14[0:3, 1])
    + 6000 * sum(x14[0:2, 2])
    + 7300 * x14[0, 3]
)
constraints14 = [
    sum(x14[0, :]) >= 15,
    sum(x14[0, 1:]) + sum(x14[1, 0:4]) >= 10,
    sum(x14[0, 2:]) + sum(x14[1, 1:3]) + sum(x14[2, 0:2]) >= 20,
    x14[0, 3] + x14[1, 2] + x14[2, 1] + x14[3, 0] >= 12,
    x14[1, 3] == 0,
    x14[2, 2] == 0,
    x14[2, 3] == 0,
    x14[3, 1] == 0,
    x14[3, 2] == 0,
    x14[3, 3] == 0,
]
problem14 = cp.Problem(objective14, constraints14)
problem14.solve()
print(f"Optimal value: {problem14.value:.0f}")
print(f"x = {np.abs(x14.value)}")

## 1.5 读取并展示示例数据

用 `pandas.read_csv` 读取 `data/data1_5.csv`，并在 notebook 末尾直接返回该 DataFrame 以表格形式展示，作为后续线性规划/数据建模示例的输入数据。

In [ ]:
data15 = pd.read_csv("data/data1_5.csv")
data15

## 1.6

## 3.3 PageRank 迭代矩阵与稳态分布

给定有向图的边表 `L`（源程序 ex3_3.py），构造 $6\times 6$ 邻接矩阵 `w` 并按行归一化得到随机转移矩阵 `P`；对 `P.T` 用稀疏特征值求解 `eigs(P.T, 1)` 取主特征向量，取实部展平后归一化为各节点的 PageRank 稳态概率 `V`，打印结果并用柱状图展示 6 个节点的得分。

In [ ]:
L = [(1, 2), (2, 3), (2, 4), (3, 4), (3, 5), (3, 6), (4, 1), (5, 6), (6, 1)]
w = np.zeros((6, 6))
for i in range(len(L)):
    w[L[i][0] - 1, L[i][1] - 1] = 1
r = np.sum(w, axis=1, keepdims=True)
P = w / np.tile(r, (1, w.shape[1]))
_, vec = eigs(P.T, 1)
V = vec.real
V = V.flatten()
V = V / V.sum()
print(f"V={V}")
fig, ax = plt.subplots()
ax.bar(range(1, len(w) + 1), V, width=0.6, color="b")
plt.show()

## 3.4 带阻尼系数的 PageRank

与 3.3 相同的图结构（源程序 ex3_4.py），在转移矩阵中加入阻尼系数 0.85：`P = (1-0.85)/n + 0.85·(w 按行归一)`，以缓解孤岛/悬挂节点导致的收敛问题；同样取 `P.T` 的主特征向量实部、展平并归一化得稳态分布 `V`，打印保留 4 位小数并用柱状图展示。

In [ ]:
L = [(1, 2), (2, 3), (2, 4), (3, 4), (3, 5), (3, 6), (4, 1), (5, 6), (6, 1)]
w = np.zeros((6, 6))
for i in range(len(L)):
    w[L[i][0] - 1, L[i][1] - 1] = 1
r = np.sum(w, axis=1, keepdims=True)
P = (1 - 0.85) / w.shape[0] + 0.85 * w / np.tile(r, (1, w.shape[1]))
_, vec = eigs(P.T, 1)
V = vec.real
V = V.flatten()
V = V / V.sum()
print("V=", np.round(V, 4))
fig, ax = plt.subplots()
ax.bar(range(1, len(w) + 1), V, width=0.6, color="b")
plt.show()

## 3.5 奇异值分解重建矩阵

对 $3\times 3$ 矩阵 `a` 调用 `svd` 得到左奇异向量 `u`、奇异值 `s`、右奇异向量转置 `vt`（源程序 ex3_5.py），并用 `u @ diag(s) @ vt` 验证可精确重建原矩阵，最后分别打印 `u`、`s`、`vt`。

In [ ]:
a = np.array([[1, 0, 1], [0, 1, 1], [0, 0, 0]])
u, s, vt = svd(a)
a = u @ np.diag(s) @ vt
print(u)
print(s)
print(vt)

## 3.6 基于相关系数的推荐系统

读取 `data/3_6_1.txt`（用户对物品的评分矩阵），用 `b = 0.5·corrcoef(a.T)+0.5` 把物品间相关系数归一化到 $[0,1]$ 得到相似度矩阵（源程序 ex3_6.py），并存为 `data/3_6_2.xlsx`。随后输入用户编号，找出其未评分项 `no` 与已评分项 `yb`，以相似度对已评分加权平均（`ys @ sim / sum(sim)`）预测未评分项的分数并打印。

In [ ]:
a = np.loadtxt("data/3_6_1.txt")
b = 0.5 * np.corrcoef(a.T) + 0.5
c = pd.DataFrame(b)
c.to_excel("data/3_6_2.xlsx", index=False)

print("请输入人员编号 1-18")
user = int(input())
n = a.shape[1]
no = np.where(a[user - 1, :] == 0)[0]
yb = set(range(n)) - set(no)
yb = list(yb)
ys = a[user - 1, yb]
sc = np.zeros(len(no))
for i in range(len(no)):
    sim = b[no[i], yb]
    sc[i] = ys @ sim / sum(sim)
print("未评分项的编号为：", no + 1)
print("未评分项的分数为：", np.round(sc, 4))

## 3.6 SVD 降维后再做基于相似度的推荐

同源数据的第二种做法（源程序 ex3_6_2.py）：先对评分矩阵做 SVD，按奇异值平方累积贡献率达到 90% 选取前 `ind` 个主成分，得到降维表示 `b`；再用列范数构造归一化相似度 `d = 0.5·(b.T b)/(c.T c)+0.5` 存为 `data/3_6_3.xlsx`。同样输入用户编号，用降维后的相似度对已评分加权，预测未评分项分数。与上一单元的区别仅在于相似度来自 SVD 降维后的协方差型公式，而非原始相关系数。

In [ ]:
a = np.loadtxt("data/3_6_1.txt")
u, sigma, vt = np.linalg.svd(a)
print(sigma)
cs = np.cumsum(sigma**2)
rate = cs / cs[-1]
ind = np.where(rate >= 0.9)[0][0] + 1
b = np.diag(sigma[:ind]) @ u.T[:ind, :] @ a

c = np.linalg.norm(b, axis=0, keepdims=True)
d = 0.5 * b.T @ b / (c.T @ c) + 0.5
dd = pd.DataFrame(d)
dd.to_excel("data/3_6_3.xlsx", index=False)

print("请输入人员编号 1-18")
user = int(input())
n = a.shape[1]
no = np.where(a[user - 1, :] == 0)[0]
yb = set(range(n)) - set(no)
yb = list(yb)
ys = a[user - 1, yb]
sc = np.zeros(len(no))
for i in range(len(no)):
    sim = d[no[i], yb]
    sc[i] = ys @ sim / sum(sim)
print("未评分项的编号为：", no + 1)
print("未评分项的分数为：", np.round(sc, 4))

## 3.7 用 SVD 做图像压缩

读取 `data/Lena.bmp` 并转为灰度浮点矩阵 `b`（源程序 ex3_7.py）；对 `b` 做 SVD 得到 `p,d,q` 与矩阵秩 `R`。先绘制奇异值随序号的散点图；再取前 $K+1$ 个奇异值/奇异向量做低秩近似重建图像，用双子图（左原图、右重建图）逐档对比，并按压缩比 $1-K(m+n+1)/(mn)$ 统计 `CR`；最后绘制“奇异值个数—压缩比率”折线图。

In [ ]:
a = Image.open("data/Lena.bmp")
if a.mode != "L":
    a = a.convert("L")
b = np.array(a).astype(float)
[p, d, q] = LA.svd(b)
m, n = b.shape
R = LA.matrix_rank(b)

fig, ax = plt.subplots()
ax.plot(np.arange(1, len(d) + 1), d, "k.")
ax.set(ylabel="奇异值", title="图像矩阵的奇异值")

CR = []
for K in range(1, int(R / 4), 10):
    fig2, (ax2, ax3) = plt.subplots(1, 2)
    ax2.set(title="原图")
    ax2.imshow(b, cmap="gray")
    recon = p[:, : K + 1] @ (np.diag(d[: K + 1])) @ (q[: K + 1, :])
    ax3.set(title="图像矩阵的秩=" + str(K))
    ax3.imshow(recon, cmap="gray")
    src = m * n
    compress = K * (m + n + 1)
    ratio = (1 - compress / src) * 100
    CR.append(ratio)
    print(f"Rank={R}:K={K}个：ratio={ratio:.2f}%")

fig3, ax4 = plt.subplots()
ax4.plot(range(1, int(R / 4), 10), CR, "ob-")
ax4.set(title="奇异值个数与压缩比率的关系", xlabel="奇异值个数", ylabel="压缩比率")
plt.show()